# MMASH sleep efficiency to mood/emotion check

This notebook tests whether **sleep efficiency** predicts MMASH PANAS mood/emotion variables.

Strict installation rule:

- Use a path only when it beats the held-out baseline mean RMSE by a meaningful amount.
- Negative test R² or tiny RMSE gains mean the path is weak or not usable.
- Random forest is kept only as a comparison, not as an installation formula.

In [1]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def rmse(y, pred):
    return float(np.sqrt(mean_squared_error(y, pred)))

df = pd.read_csv('mmash_bridge_data.csv')
targets = [c for c in df.columns if c.startswith('panas_')]
rows = []
for target in targets:
    d = df[['Efficiency', target]].dropna()
    if len(d) < 12:
        continue
    X = d[['Efficiency']]
    y = d[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
    baseline_pred = np.full(len(y_test), y_train.mean())
    linear_model = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
    forest_model = RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=3).fit(X_train, y_train)
    linear_pred = linear_model.predict(X_test)
    forest_pred = forest_model.predict(X_test)
    baseline_rmse = rmse(y_test, baseline_pred)
    model_options = [
        ('linear regression', linear_pred, linear_model.score(X_train, y_train)),
        ('random forest', forest_pred, forest_model.score(X_train, y_train)),
    ]
    best_name, best_pred, best_train_r2 = min(model_options, key=lambda item: rmse(y_test, item[1]))
    best_rmse = rmse(y_test, best_pred)
    best_r2 = r2_score(y_test, best_pred)
    improvement = (baseline_rmse - best_rmse) / baseline_rmse if baseline_rmse else 0
    if improvement >= 0.05 and best_r2 > 0 and best_name == 'linear regression':
        decision = 'usable'
    elif best_rmse < baseline_rmse:
        decision = 'weak / exploratory only'
    else:
        decision = 'do not use'
    lr = linear_model.named_steps['linearregression']
    scaler = linear_model.named_steps['standardscaler']
    raw_coef = lr.coef_[0] / scaler.scale_[0]
    raw_intercept = lr.intercept_ - raw_coef * scaler.mean_[0]
    rows.append({
        'path': f'Sleep efficiency → {target}',
        'output': 'mood/emotion',
        'target': target,
        'predictors': 'Efficiency',
        'rows_used': len(d),
        'baseline_r2': round(r2_score(y_test, baseline_pred), 4),
        'baseline_mae': round(mean_absolute_error(y_test, baseline_pred), 4),
        'baseline_rmse': round(baseline_rmse, 4),
        'linear_train_r2': round(linear_model.score(X_train, y_train), 4),
        'linear_test_r2': round(r2_score(y_test, linear_pred), 4),
        'linear_mae': round(mean_absolute_error(y_test, linear_pred), 4),
        'linear_rmse': round(rmse(y_test, linear_pred), 4),
        'rf_train_r2': round(forest_model.score(X_train, y_train), 4),
        'rf_test_r2': round(r2_score(y_test, forest_pred), 4),
        'rf_mae': round(mean_absolute_error(y_test, forest_pred), 4),
        'rf_rmse': round(rmse(y_test, forest_pred), 4),
        'best_model': best_name if best_rmse < baseline_rmse else 'baseline mean',
        'rmse_improvement_pct': round(100 * improvement, 2),
        'decision': decision,
        'raw_formula': f'{target} = {raw_intercept:.4f} {raw_coef:+.4f} * Efficiency',
        'standardized_formula': f'{target} = {lr.intercept_:.4f} {lr.coef_[0]:+.4f} * z(Efficiency)',
    })

results = pd.DataFrame(rows)
results.to_csv('mmash_sleep_efficiency_mood_results.csv', index=False)
results


,path,output,target,predictors,rows_used,baseline_r2,baseline_mae,baseline_rmse,linear_train_r2,linear_test_r2,...,linear_rmse,rf_train_r2,rf_test_r2,rf_mae,rf_rmse,best_model,rmse_improvement_pct,decision,raw_formula,standardized_formula
0,Sleep efficiency → panas_pos_10,mood/emotion,panas_pos_10,Efficiency,21,-0.0358,1.5816,2.6888,0.0045,0.0311,...,2.6004,0.1601,0.1346,1.5875,2.4576,random forest,8.60,weak / exploratory only,panas_pos_10 = 31.7288 -0.0493 * Efficiency,panas_pos_10 = 27.6429 -0.3573 * z(Efficiency)
1,Sleep efficiency → panas_pos_14,mood/emotion,panas_pos_14,Efficiency,20,-1.5762,5.2143,6.6662,0.1230,-3.8360,...,9.1335,0.1547,-2.5850,6.5613,7.8639,baseline mean,-17.97,do not use,panas_pos_14 = 57.8669 -0.3734 * Efficiency,panas_pos_14 = 25.7143 -2.0249 * z(Efficiency)
2,Sleep efficiency → panas_pos_18,mood/emotion,panas_pos_18,Efficiency,21,-0.0673,6.7245,7.6801,0.1211,-0.0915,...,7.7666,0.2912,-0.1495,6.2672,7.9705,baseline mean,-1.13,do not use,panas_pos_18 = 3.6316 +0.2708 * Efficiency,panas_pos_18 = 26.0714 +1.9621 * z(Efficiency)
3,Sleep efficiency → panas_pos_22,mood/emotion,panas_pos_22,Efficiency,21,-0.1419,3.3571,3.8499,0.0015,-0.1377,...,3.8427,0.2487,-0.3275,2.9934,4.1510,linear regression,0.19,weak / exploratory only,panas_pos_22 = 22.7608 -0.0273 * Efficiency,panas_pos_22 = 20.5000 -0.1977 * z(Efficiency)
4,Sleep efficiency → panas_pos_9+1,mood/emotion,panas_pos_9+1,Efficiency,21,-0.1931,4.2347,5.5037,0.0001,-0.1902,...,5.4970,0.1050,-0.1546,3.8551,5.4140,random forest,1.63,weak / exploratory only,panas_pos_9+1 = 25.1926 -0.0066 * Efficiency,panas_pos_9+1 = 24.6429 -0.0481 * z(Efficiency)
5,Sleep efficiency → panas_neg_10,mood/emotion,panas_neg_10,Efficiency,21,-0.0022,3.1735,4.5631,0.0504,-0.0217,...,4.6072,0.4545,-0.2275,4.7767,5.0500,baseline mean,-0.97,do not use,panas_neg_10 = 1.9660 +0.1461 * Efficiency,panas_neg_10 = 14.0714 +1.0585 * z(Efficiency)
6,Sleep efficiency → panas_neg_14,mood/emotion,panas_neg_14,Efficiency,20,-0.0386,3.0714,4.0764,0.0905,-0.6188,...,5.0893,0.2697,-0.5550,3.5145,4.9879,baseline mean,-22.36,do not use,panas_neg_14 = -8.3453 +0.2620 * Efficiency,panas_neg_14 = 14.2143 +1.4207 * z(Efficiency)
7,Sleep efficiency → panas_neg_18,mood/emotion,panas_neg_18,Efficiency,21,-1.0622,5.1429,6.3696,0.0893,-1.1220,...,6.4613,0.3568,-1.1515,5.1510,6.5059,baseline mean,-1.44,do not use,panas_neg_18 = 23.9194 -0.1197 * Efficiency,panas_neg_18 = 14.0000 -0.8673 * z(Efficiency)
8,Sleep efficiency → panas_neg_22,mood/emotion,panas_neg_22,Efficiency,21,-0.2353,3.7347,5.5641,0.0279,-0.3122,...,5.7346,0.2315,-0.3408,4.1812,5.7968,baseline mean,-3.06,do not use,panas_neg_22 = 8.5329 +0.0694 * Efficiency,panas_neg_22 = 14.2857 +0.5030 * z(Efficiency)
9,Sleep efficiency → panas_neg_9+1,mood/emotion,panas_neg_9+1,Efficiency,21,-0.0198,1.8776,2.0504,0.0792,0.0005,...,2.0299,0.3332,-0.1858,2.0026,2.2110,linear regression,1.00,weak / exploratory only,panas_neg_9+1 = 21.1077 -0.0961 * Efficiency,panas_neg_9+1 = 13.1429 -0.6964 * z(Efficiency)


## Decision

The MMASH PANAS paths are treated as **exploratory only or unusable** because the sample is very small and the interpretable linear models do not provide a strong, stable held-out improvement.